# 🐻🐂 Bull-Bear Debate Stock Analysis

Run a multi-round **LangGraph** debate between a **Bear** and a **Bull** researcher, judged by an impartial **Judge**, for any company.

This notebook installs dependencies, uploads the source as a ZIP, configures your API key, and runs a debate end-to-end. Tools are mocked in V1; the debate/agent orchestration is the focus.


In [ ]:
#@title 1. Install dependencies
%%capture
!pip install -q langgraph langchain-openai langchain-core tenacity pydantic fastapi uvicorn nest-asyncio
print("Dependencies installed.")

In [ ]:
#@title 2. Upload the source code (ZIP)
# ---------------------------------------------------------------------------
# STEP 1 (on your local machine): build the zip, e.g.
#     cd /path/to/bear_bull_debate
#     python tools/build_bear_bull_debate_zip.py
#   (or: zip -r bear_bull_debate_src.zip src/bear_bull_debate)
#   Zipping the whole project folder also works — the notebook auto-locates
#    the package inside the archive.)
#
# STEP 2: run this cell and choose the .zip file when the upload dialog appears.
# ---------------------------------------------------------------------------
from google.colab import files
import zipfile, os, sys, glob

EXTRACT_DIR = "/content/bear_bull_debate_src"
os.makedirs(EXTRACT_DIR, exist_ok=True)

uploaded = files.upload()  # choose your .zip

zipped = [name for name in uploaded if name.endswith(".zip")]
if not zipped:
    raise SystemExit("No .zip file was uploaded.")

for name in zipped:
    with zipfile.ZipFile(name, "r") as z:
        z.extractall(EXTRACT_DIR)
    print(f"Extracted {name}")

# Locate the `bear_bull_debate` package dir (handles nesting from zipping the whole project).
matches = glob.glob(os.path.join(EXTRACT_DIR, "**", "bear_bull_debate"), recursive=True)
if not matches:
    raise SystemExit("Could not find the 'bear_bull_debate' package in the uploaded zip.")

src_root = os.path.dirname(matches[0])
if src_root not in sys.path:
    sys.path.insert(0, src_root)

import bear_bull_debate
print("Source ready:", bear_bull_debate.__file__)

In [ ]:
#@title 3. Configure your API key
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # Optional: for OpenAI-compatible providers (DeepSeek / Qwen-DashScope),
    # add a Colab secret named OPENAI_BASE_URL.
    try:
        os.environ["OPENAI_BASE_URL"] = userdata.get("OPENAI_BASE_URL")
    except Exception:
        pass
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    print("API key set from prompt.")

# Optional: override per-node models for OpenAI-compatible providers.
# os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"
# os.environ["BEAR_MODEL"] = "deepseek-chat"
# os.environ["BULL_MODEL"] = "deepseek-chat"
# os.environ["JUDGE_MODEL"] = "deepseek-chat"
# os.environ["SUMMARY_MODEL"] = "deepseek-chat"

In [ ]:
#@title 4. Run a debate
import nest_asyncio
nest_asyncio.apply()

from bear_bull_debate.runner import run_debate

COMPANY = "AAPL"      # @param {type:"string"}
MAX_ROUNDS = 2        # @param {type:"integer"}

result = run_debate(COMPANY, max_rounds=MAX_ROUNDS)

print("Thread ID:", result["thread_id"])
print("\n" + "=" * 70)
print(result["final_report"])
print("=" * 70)
print("\nTool calls performed:")
for log in result["tool_logs"]:
    print("  -", log)

## Customize

- **Models** — set `BEAR_MODEL`, `BULL_MODEL`, `JUDGE_MODEL`, `SUMMARY_MODEL` env vars.
- **OpenAI-compatible providers** — set `OPENAI_BASE_URL` (e.g. `https://api.deepseek.com`) and use that provider's model names.
- **Rounds** — change `MAX_ROUNDS` (1–5).
- **Async** — use `await run_debate_async(...)` instead (IPython supports top-level await, so `nest_asyncio` is optional).
